# Tokenizers: The Bridge Between English and Math

Neural networks, including Large Language Models (LLMs) and Diffusion models, cannot read English. They only understand numbers (tensors/matrices).

Before we can send a prompt to an AI model, we must translate our text into a mathematical format. A **Tokenizer** is the dictionary and the engine that performs this exact translation. It chops our text into pieces, looks up each piece in a massive dictionary, and replaces it with a unique integer ID.

## 1. The Core Problem

Historically, early AI researchers tried two different methods to translate text, both of which failed for massive models:

**Attempt A: Word-Level Tokenization**
* **How it works:** Assign a unique number to every single word in the English language.
* **The Problem:** The dictionary gets too big. What about typos? What about new words like "crypto" or "COVID"? If the model sees a word it doesn't have in its dictionary, it replaces it with a `<UNK>` (Unknown) token, instantly losing the meaning of the sentence.

**Attempt B: Character-Level Tokenization**
* **How it works:** Assign a unique number to the 26 letters of the alphabet, plus punctuation.
* **The Problem:** It loses semantic meaning. The letter "c" means nothing on its own. Furthermore, a single sentence becomes hundreds of numbers, destroying the model's Context Window limits and causing Out of Memory (OOM) errors.

## 2. Subword Tokenization (The Modern Standard)

Modern models (like GPT-4, Llama 3, and BERT) use **Subword Tokenization**. This is the "Goldilocks" solution.

Instead of chopping by whole words or individual letters, it chops text by *common syllables and word fragments*.

**How it works (Byte-Pair Encoding / WordPiece):**
It breaks down complex or unknown words into smaller, known logical chunks.
* Example: The word `"unbelievably"` might not be in the dictionary.
* The tokenizer splits it into: `["un", "##believ", "##ably"]`
* *Note: The `##` (or sometimes a `Ġ` symbol) just tells the model that this piece attaches directly to the piece before it without a space.*

**The Result:** The model never gets an `<UNK>` (Unknown) error! If it sees a completely made-up word, it will just chop it all the way down to its base letters if it has to. It forces the model to understand the *roots* of words, making it incredibly smart at deducing the meaning of new vocabulary.

## 3. The Hugging Face Tokenization Pipeline

When you call a Hugging Face `AutoTokenizer`, it secretly runs your text through four distinct steps:

1. **Pre-processing / Normalization:** It cleans the text (e.g., removing extra spaces, standardizing accents, sometimes lowercasing).
2. **Pre-tokenization:** It splits the sentence into raw words based on spaces and punctuation.
3. **Tokenization:** It applies the Subword algorithm (BPE/WordPiece) to chop the words into the final subword chunks.
4. **Post-processing:** This is crucial! It adds **Special Tokens** that the specific neural network requires to understand where the prompt starts and ends.
   * Example: BERT requires a `[CLS]` (Classification) token at the start and a `[SEP]` (Separator) token at the end.
   * Example: Llama requires a `<|begin_of_text|>` token.

In [2]:
# Imports:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer

In [3]:
# Connecting to Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')

if hf_token and hf_token.startswith('hf_'):
    print('Hugging Face Token Found')
    login(token= hf_token, add_to_git_credential= True)
    print('Successfully logged into Hugging Face')
else:
    print('Hugging Face Token not found')

Hugging Face Token Found


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully logged into Hugging Face


In [4]:
# Checking GPU Access:

import torch

gpu_availability = torch.cuda.is_available()
if gpu_availability:
    print('GPU Available')
    device = torch.cuda.get_device_name()
    print(device)
else:
    print('GPU not available')

GPU Available
NVIDIA GeForce RTX 4080 Laptop GPU


## Accessing Llama 3.1 from Meta

In order to use the fantastic Llama 3.1, Meta does require you to sign their terms of service.

Visit their model instructions page in Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

At the top of the page are instructions on how to agree to their terms. If possible, you should use the same email as your huggingface account.

In my experience approval comes in a couple of minutes. Once you've been approved for any 3.1 model, it applies to the whole 3.1 family of models. For whatever reason, occasionally Meta doesn't approve access. If that happens to you, please follow [this](https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8?usp=sharing) troubleshooting.

If the next cell gives you an error, then please check:
1. Are you logged in to HuggingFace? Try running `login()` to check your key works
2. Did you set up your API key with full read and write permissions?
3. If you visit the Llama3.1 page with the link above, does it show that you have access to the model near the top?

I've also set up this troubleshooting colab to try to diagnose any HuggingFace connectivity issues:
https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8?usp=sharing

In [5]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code= True)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3.1-8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [6]:
# Encoding Text into Tokens:

text = 'I am excited to show Tokenizers in action to my LLM engineers'
tokens = tokenizer.encode(text)
print(tokens)

[128000, 40, 1097, 12304, 311, 1501, 9857, 12509, 304, 1957, 311, 856, 445, 11237, 25175]


In [7]:
# Character Count, Word Count and Token Count:

character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f'There are {character_count} characters, {word_count} words, and {token_count} tokens.')

There are 61 characters, 12 words, and 15 tokens.


In [9]:
# Decoding Tokens into Text:
tokenizer.decode(tokens)

'<|begin_of_text|>I am excited to show Tokenizers in action to my LLM engineers'

In [10]:
tokenizer.batch_decode(tokens)

['<|begin_of_text|>I am excited to show Tokenizers in action to my LLM engineers']

In [12]:
# To see Special Token IDs:
tokenizer.get_added_vocab()

{'<|begin_of_text|>': 128000,
 '<|end_of_text|>': 128001,
 '<|reserved_special_token_0|>': 128002,
 '<|reserved_special_token_1|>': 128003,
 '<|finetune_right_pad_id|>': 128004,
 '<|reserved_special_token_2|>': 128005,
 '<|start_header_id|>': 128006,
 '<|end_header_id|>': 128007,
 '<|eom_id|>': 128008,
 '<|eot_id|>': 128009,
 '<|python_tag|>': 128010,
 '<|reserved_special_token_3|>': 128011,
 '<|reserved_special_token_4|>': 128012,
 '<|reserved_special_token_5|>': 128013,
 '<|reserved_special_token_6|>': 128014,
 '<|reserved_special_token_7|>': 128015,
 '<|reserved_special_token_8|>': 128016,
 '<|reserved_special_token_9|>': 128017,
 '<|reserved_special_token_10|>': 128018,
 '<|reserved_special_token_11|>': 128019,
 '<|reserved_special_token_12|>': 128020,
 '<|reserved_special_token_13|>': 128021,
 '<|reserved_special_token_14|>': 128022,
 '<|reserved_special_token_15|>': 128023,
 '<|reserved_special_token_16|>': 128024,
 '<|reserved_special_token_17|>': 128025,
 '<|reserved_special_to

In [14]:
# Length of Vocabulary of Tokenizer:
len(tokenizer.vocab)

128256

## Instruct variants of models

Many models have a variant that has been trained for use in Chats.
These are typically labeled with the word "Instruct" at the end.
They have been trained to expect prompts with a particular format that includes system, user and assistant prompts.

There is a utility method `apply_chat_template` that will convert from the messages list format we are familiar with, into the right input prompt for this model.

In [15]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

C:\Users\shail\anaconda3\envs\applied_llm_engineering\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shail\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3.1-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [16]:
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': 'Tell a light-hearted joke for a room of Data Scientists.'}
]

# Turning above messages to input prompt for the model:
prompt = tokenizer.apply_chat_template(messages,
                                       tokenize= False,
                                       add_generation_prompt= True)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## Crucial "Aha" moment

Till now, we've been given the impression that LLMs could receive a list of python dictionaries in some way:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
```

But an LLM is just a Data Science model that takes a sequence of numbers and predicts the probability of the next number! You can't pass a bunch of Python objects into a statistical model!

### And now you have the missing piece of the puzzle..

The messages in OpenAI format get converted:

1. ...into a sequence of words with special tags to separate the System, User, Assistant prompt
2. ...then the words are broken down into fragments - "tokens"
3. ...then the tokens are replaced with Token IDs - and this is the input sequence

> The input to an LLM is a sequence of Token IDs. The output is the probability distribution of the next Token ID to follow this input.

That's it!

In [ ]:
|